# Section 2: SQL - Querying and Optimization  

    • Dataset Specifications: Two tables: Sales (SaleID, Date, Region, ProductID, Quantity, Revenue; ~800 rows) and Product Categories (ProductID, ProductName, Category e.g., Electronics, Apparel; ~20 rows); supports JOINs, subqueries, and indexing.  
    • Tasks:  
        1. Write a query to calculate total revenue per region and product category, using a JOIN with the secondary table. Include a subquery to identify regions with above-average revenue.  
        2. Optimize the query by creating an index on the revenue column and explain the performance improvement. 
    • Output: SQL query file (.sql), screenshot of results, and screenshot of index creation.  
    • Interpretation: In 100-150 words, discuss the query results, the impact of indexing, and their relevance for sales strategies. 

In [1]:
import sqlite3
import pandas as pd

In [2]:
# Load the CSV
df = pd.read_csv("sales data.csv")
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Cost,Price,Profit,Quantity,Sales
0,1,CA-2017-152156,8/11/17,11/11/17,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,464.48,901.06,436.58,4,3604.243977
1,2,CA-2017-152156,8/11/17,11/11/17,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",756.10,138.70,-617.40,12,1664.369269
2,3,CA-2017-138688,12/6/17,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,537.68,159.28,-378.40,12,1911.397750
3,4,US-2016-108966,11/10/16,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,875.91,445.88,-430.03,13,5796.463018
4,5,US-2016-108966,11/10/16,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,187.17,137.59,-49.58,19,2614.284944
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9795,9796,CA-2017-125920,21/05/2017,28/05/2017,Standard Class,SH-19975,Sally Hughsby,Corporate,United States,Chicago,...,Central,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",390.95,716.41,325.46,6,4298.456294
9796,9797,CA-2016-128608,12/1/16,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,...,East,OFF-AR-10001374,Office Supplies,Art,"BIC Brite Liner Highlighters, Chisel Tip",377.87,671.92,294.05,3,2015.759993
9797,9798,CA-2016-128608,12/1/16,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,...,East,TEC-PH-10004977,Technology,Phones,GE 30524EE4,296.25,419.67,123.42,23,9652.397717
9798,9799,CA-2016-128608,12/1/16,17/01/2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,...,East,TEC-PH-10000912,Technology,Phones,Anker 24W Portable Micro USB Car Charger,60.04,95.60,35.56,4,382.392219


In [3]:
# Connect to SQLite DB
conn = sqlite3.connect("sales.db")

# Write the DataFrame to a new table in SQLite
df.to_sql("Sales", conn, if_exists="replace", index=False)

# Check table structure
pd.read_sql_query("PRAGMA table_info(Sales);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Row ID,INTEGER,0,None,0
1,1,Order ID,TEXT,0,None,0
2,2,Order Date,TEXT,0,None,0
3,3,Ship Date,TEXT,0,None,0
4,4,Ship Mode,TEXT,0,None,0
5,5,Customer ID,TEXT,0,None,0
6,6,Customer Name,TEXT,0,None,0
7,7,Segment,TEXT,0,None,0
8,8,Country,TEXT,0,None,0
9,9,City,TEXT,0,None,0


In [4]:
query = """
WITH RegionSales AS (
    SELECT 
        Region,
        SUM(Sales) AS total_region_sales
    FROM Sales
    GROUP BY Region
),
AboveAvgRegions AS (
    SELECT 
        Region
    FROM RegionSales
    WHERE total_region_sales > (
        SELECT AVG(total_region_sales) FROM RegionSales
    )
)
SELECT 
    s.Region,
    s.Category,
    SUM(s.Sales) AS Total_Revenue
FROM Sales s
JOIN AboveAvgRegions r
    ON s.Region = r.Region
GROUP BY s.Region, s.Category
ORDER BY s.Region, s.Category;
"""

df_result = pd.read_sql_query(query, conn)
df_result

,Region,Category,Total_Revenue
0,East,Furniture,3.823634e+06
1,East,Office Supplies,1.095057e+07
2,East,Technology,3.635478e+06
3,West,Furniture,4.285769e+06
4,West,Office Supplies,1.152568e+07
5,West,Technology,4.155224e+06


In [5]:
# Create index
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales ON Sales(Sales);")
conn.commit()

In [6]:
%timeit pd.read_sql_query(query, conn)

14.2 ms ± 527 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
with open("denaco_task2_query.sql", "w") as f:
    f.write(query)